# D1-04 First LCA from demand to score in Brightway

This notebook is the main Day 1 milestone: we bootstrap the course project from a prepared Brightway archive, import the bundled BAFU workbook, inspect the resulting database, and run a first LCA.

The hands-on exercises use BAFU throughout the course. We use `ecoinvent-3.10-biosphere` only as a convenient starting project because it already contains `biosphere3` and a populated set of LCIA methods.


## Learning goals

- Install or switch to the course `brightway` project from a prepared archive.
- Import the bundled BAFU workbook with `bw2io.ExcelImporter`.
- Understand the role of strategies, biosphere matching, statistics, and database writing.
- Inspect the LCIA methods that are already available in the project.
- Inspect the imported database by searching for activities and exchanges.
- Run a first LCA score with an explicitly chosen activity and method.
- Recognize how an `ecoinvent` import would fit into the same overall workflow.


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Database of the Swiss Federal Administration, FOEN:20XY, Federal Office for the Environment, 2025.


## 1) Install or switch to the course project

We start from a prepared Brightway project archive instead of building `biosphere3` and the LCIA methods from scratch.

The call `bi.remote.install_project('ecoinvent-3.10-biosphere', project_name)` downloads the archive on first use, restores it as a new project, and gives us a project that already contains `biosphere3` and many LCIA methods.
If the project already exists locally, we simply switch to it.


In [1]:
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import bw2calc as bc
import bw2data as bd
import bw2io as bi

# this is just to let dataframe print fully
pd.set_option('display.max_colwidth', None)

/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [6]:
# let's choose a project name
project_name = "barcelona-rlcia-2026"

# and make a list of existing projects
existing_projects = [project.name for project in bd.projects]

In [7]:
# if the project is already part of the projects' list, then our project already exists
# in this case, we only need to "switch" to it
if project_name in existing_projects:
    bd.projects.set_current(project_name)
    print('Switched to existing project:', bd.projects.current)

# if not, then we create it
else:
    bi.remote.install_project(project_key, project_name)
    # and we switch to it
    bd.projects.set_current(project_name)
    print('Installed and switched to project:', bd.projects.current)

Switched to existing project: barcelona-rlcia-2026


In [8]:
# this is needed later when importing LCI databases
bi.create_core_migrations()

print('Databases available now:', list(bd.databases))
print('LCIA methods available now:', len(bd.methods))

Databases available now: ['ecoinvent-3.10-biosphere', 'bafu', 'bafu-notebook-check']
LCIA methods available now: 668


Let's check the size of `ecoinvent-3.10-biosphere`

In [9]:
len(bd.Database("ecoinvent-3.10-biosphere"))

4362

The prepared project already contains the core biosphere database and a large set of LCIA methods.
So the hands-on import work in this notebook is focused on the BAFU technosphere database.


## 2) Confirm the bundled BAFU workbook

The BAFU database we will be using is available as a workbook, contained in this repository:

- `data/lci-bafu.xlsx`


In [19]:
bafu_file = Path('../../data/lci-bafu.xlsx')
print('Workbook path:', bafu_file.resolve())
print('Found:', bafu_file.exists())
if bafu_file.exists():
    print('Workbook size [MB]:', round(bafu_file.stat().st_size / 1024 / 1024, 2))


Workbook path: /Users/romain/GitHub/barcelona-regionalized-lcia/data/lci-bafu.xlsx
Found: True
Workbook size [MB]: 29.38


## 3) Import pipeline: `ExcelImporter` -> strategies -> biosphere matching -> statistics -> database writing

This is the import sequence to understand in this notebook:

1. Create an `ExcelImporter` object from the workbook path.
2. Apply the standard import strategies (data cleanup, mostly).
3. Match biosphere flows against `biosphere3`.
4. Check import statistics and unlinked exchanges.
5. Write the database to the current project.


In [31]:
database_name = 'bafu'

# create the ExcelImporter object
importer = bi.ExcelImporter(str(bafu_file))
print('Importer type:', type(importer).__name__)
print('Raw datasets loaded:', len(importer.data))


Extracted 1 worksheets in 38.02 seconds
Importer type: ExcelImporter
Raw datasets loaded: 11747


In [21]:
# data cleanup
importer.apply_strategies()
print('Applied import strategies.')

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 4.56 seconds
Applied import strategies.


In [22]:
importer.match_database(fields=('name', 'reference product', 'location'))

Applying strategy: link_iterable_by_fields


In [23]:
# now, we try to match the biosphere exchanges with the flows of the `biosphere` database
# on the basis of 'name', 'unit', 'categories'
importer.match_database('ecoinvent-3.10-biosphere', fields=('name', 'unit', 'categories'))
print('Matched biosphere exchanges against biosphere3.')

Applying strategy: link_iterable_by_fields
Matched biosphere exchanges against biosphere3.


In [24]:
# we check the status of exchanges linking
importer.statistics()

Graph statistics for `bafu` importer:
11747 graph nodes:
	processwithreferenceproduct: 11747
408975 graph edges:
	biosphere: 284445
	technosphere: 112783
	production: 11747
408975 edges to the following databases:
	ecoinvent-3.10-biosphere: 284445
	bafu: 124530
0 unique unlinked edges (0 total):




(11747, 408975, 0, 0)

`0 unique unlinked edges (0 total)` is usually the sweet message you want to see.  
This means we can go ahead and write the database to the project.

In [25]:
importer.write_database()

20:44:10+0100 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|████████████████████████████████████| 11747/11747 [00:19<00:00, 604.48it/s]

20:44:29+0100 [info     ] Vacuuming database            


Created database: bafu
Created database: bafu-2025


## 4) Inspect the LCIA methods already available in the project

Because the project archive came from `ecoinvent-3.10-biosphere`, the LCIA methods are already in place.
So, unlike the previous version of this notebook, we do not import LCIA methods from local Excel files here.


In [26]:
preferred_method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

print('Databases available:', list(bd.databases))
print('Number of LCIA methods available:', len(bd.methods))
print('Preferred climate-change method available:', preferred_method in bd.methods)


Databases available: ['ecoinvent-3.10-biosphere', 'bafu']
Number of LCIA methods available: 668
Preferred climate-change method available: True


Let's quickly inspect a few available methods before choosing one for the first LCA.


In [27]:
sample_methods = sorted(bd.methods)[:10]
climate_methods = [m for m in sorted(bd.methods) if 'climate change' in ' | '.join(m).lower()]

print('First 10 LCIA methods in the project:')
for method in sample_methods:
    print('-', ' | '.join(method))

print('\nFirst climate-related methods:')
for method in climate_methods[:10]:
    print('-', ' | '.join(method))


First 10 LCIA methods in the project:
- CML v4.8 2016 | acidification | acidification (incl. fate, average Europe total, A&B)
- CML v4.8 2016 | climate change | global warming potential (GWP100)
- CML v4.8 2016 | ecotoxicity: freshwater | freshwater aquatic ecotoxicity (FAETP inf)
- CML v4.8 2016 | ecotoxicity: marine | marine aquatic ecotoxicity (MAETP inf)
- CML v4.8 2016 | ecotoxicity: terrestrial | terrestrial ecotoxicity (TETP inf)
- CML v4.8 2016 | energy resources: non-renewable | abiotic depletion potential (ADP): fossil fuels
- CML v4.8 2016 | eutrophication | eutrophication (fate not incl.)
- CML v4.8 2016 | human toxicity | human toxicity (HTP inf)
- CML v4.8 2016 | material resources: metals/minerals | abiotic depletion potential (ADP): elements (ultimate reserves)
- CML v4.8 2016 | ozone depletion | ozone layer depletion (ODP steady state)

First climate-related methods:
- CML v4.8 2016 | climate change | global warming potential (GWP100)
- CML v4.8 2016 no LT | climate ch

### REMINDER: how to inspect an LCIA method?

In [11]:
my_method = bd.methods.random()
my_method

('Inventory results and indicators', 'emissions to air', 'SOx')

In [13]:
method_obj = bd.Method(my_method)
method_data = method_obj.load()

In [17]:
print('Method metadata:')
pprint(method_obj.metadata)

Method metadata:
{'abbreviation': 'inventory-results-and-indicatorses.3e20c2c7b552fd2323ea796fa4a7922a',
 'database': 'ecoinvent-3.10-biosphere',
 'ecoinvent_version': '3.10',
 'filepath': '/Users/cmutel/Library/Application '
             'Support/EcoinventInterface/cache/ecoinvent '
             '3.10_LCIA_implementation/LCIA Implementation 3.10.xlsx',
 'geocollections': ['world'],
 'num_cfs': 10,
 'unit': 'kg'}


In [19]:
# an empty list, to store the results
lcia_data = []

for flow_id, cf in method_data:
    flow = bd.get_activity(flow_id)
    lcia_data.append(
        [
            flow["name"],
            cf
        ]
    )

pd.DataFrame(lcia_data, columns=["flow name", "cf"])

,flow name,cf
0,Sulfur dioxide,1.0
1,Sulfur dioxide,1.0
2,Sulfur dioxide,1.0
3,Sulfur dioxide,1.0
4,Sulfur dioxide,1.0
5,Sulfur oxides,1.0
6,Sulfur trioxide,1.0
7,Sulfur trioxide,1.0
8,Sulfur trioxide,1.0
9,Sulfur trioxide,1.0


## 5) Import your own LCIA method

We will use bw2io.LCIAExcelImporter to import a user-defined LCIA method file.  
You can find those files under `data/LCIA methods`. We will import `01__acidification__accumulated_exceedance_ae.xlsx`.

In [33]:
importer = bi.ExcelLCIAImporter(
    "../../data/LCIA methods/01__acidification__accumulated_exceedance_ae.xlsx",
    name=("EF 3.0", "Acidification"),
    description="Some method about ocean acidification",
    unit="kg SO2-eq."
)

In [34]:
# data cleanup
importer.apply_strategies()

Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.06 seconds


In [37]:
# let's check if we have any unlinked CFs
importer.statistics()

1 methods
19 cfs
0 unlinked cfs


(1, 19, 0)

In [36]:
# we're good to go!
importer.write_methods()

Wrote 1 LCIA methods with 19 characterization factors


In [38]:
# let's confirm that our method is available
("EF 3.0", "Acidification") in bd.methods

True

In [40]:
# let's inspect it
acid_method = bd.Method(("EF 3.0", "Acidification"))
pprint(acid_method.metadata)

{'abbreviation': 'ef-30a.33e899d02d0b2714248f9d3f8c74391d',
 'description': 'Some method about ocean acidification',
 'filename': '01__acidification__accumulated_exceedance_ae.xlsx',
 'geocollections': ['world'],
 'num_cfs': 19,
 'unit': 'kg SO2-eq.'}


In [41]:
for flow_id, cf in acid_method.load():
    flow = bd.get_activity(flow_id)
    print(flow["name"], cf)

Ammonia 3.02
Nitrogen oxides 0.74
Nitrogen oxides 0.74
Ammonia 3.02
Sulfur trioxide 1.04821
Sulfur trioxide 1.04821
Sulfur trioxide 1.04821
Sulfur dioxide 1.31
Sulfur dioxide 1.31
Ammonia 3.02
Nitrogen oxides 0.74
Sulfur dioxide 1.31
Sulfur dioxide 1.31
Nitrogen oxides 0.74
Nitrogen oxides 0.74
Sulfur trioxide 1.04821
Sulfur oxides 1.31
Nitric oxide 1.13467
Ammonia 3.02


## 6) Inspect the imported database

Now we reconnect the import workflow from the previous notebooks: project -> database -> activity -> exchange.

A good way to start is to search the database by name, then refine with additional filters such as `location`.

In [ ]:


starter_queries = [
    'operation, passenger car, petrol',
    'passenger car, gasoline',
    'gasoline',
]
starter_hits = []
used_query = None

for query in starter_queries:
    hits = db.search(query, limit=10)
    if hits:
        starter_hits = hits
        used_query = query
        break

print('Search query used:', used_query)

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in starter_hits
    ]
)

In [43]:
db = bd.Database("bafu")
print('Number of activities in bafu:', len(db))

Number of activities in bafu: 11747


We can use `bw2data.search()` to look for activities (this, naturally, works also to search the `biosphere` database).  
You can always display the arguments and teh docstrings of a function like so:

In [51]:
db.search?
# db.search() is quite complex if it needs to be...

Signature: db.search(string, **kwargs)
Docstring:
Search this database for ``string``.

The searcher include the following fields:

* name
* comment
* categories
* location
* reference product

``string`` can include wild cards, e.g. ``"trans*"``.

By default, the ``name`` field is given the most weight. The full weighting set is called the ``boost`` dictionary, and the default weights are::

    {
        "name": 5,
        "comment": 1,
        "product": 3,
        "categories": 2,
        "location": 3
    }

Optional keyword arguments:

* ``limit``: Number of results to return.
* ``boosts``: Dictionary of field names and numeric boosts - see default boost values above. New values must be in the same format, but with different weights.
* ``filter``: Dictionary of criteria that search results must meet, e.g. ``{'categories': 'air'}``. Keys must be one of the above fields.
* ``mask``: Dictionary of criteria that exclude search results. Same format as ``filter``.
* ``facet``: Field to

In [54]:
db.search(
    "gasoline",
    filter={
        "location": "RER",
        "reference product": "car"
    },
    limit=10
)

['Transport, passenger car, gasoline, fleet average' (kilometer, RER, None),
 'Transport, passenger car, gasoline, fleet average' (kilometer, CH, None),
 'Transport, Motorbike, gasoline, 2020 fleet average' (kilometer, CH, None),
 'Fuel supply for gasoline vehicles' (kilogram, RER, None),
 'Fuel supply for gasoline vehicles' (kilogram, CH, None),
 'Passenger car, gasoline, Medium, 2003, EURO-3' (unit, RER, None),
 'Passenger car, gasoline, Compact, 2008, EURO-4' (unit, RER, None),
 'Passenger car, gasoline, Medium, 2008, EURO-4' (unit, RER, None),
 'Passenger car, gasoline, Compact, 2003, EURO-3' (unit, RER, None),
 'Passenger car, gasoline, Compact, 2018, EURO-6c' (unit, RER, None)]

A `search` gets you close quickly. To be more explicit, you can iterate through the database yourself and combine several criteria in a list comprehension.

In [59]:
gasoline_car_candidates = [
    act
    for act in db
    if 'gasoline' in act['name'].lower()
    and any(term in act['reference product'].lower() for term in ['car', 'gasoline'])
    and act["location"] == "RER"
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in gasoline_car_candidates[:10]
    ]
)

,name,location,unit
0,"Transport, passenger car, plugin gasoline hybr...",RER,kilometer
1,"Transport, passenger car, plugin gasoline hybr...",RER,kilometer
2,"Transport, passenger car, gasoline, Compact, 2...",RER,kilometer
3,"Transport, passenger car, gasoline hybrid, Com...",RER,kilometer
4,"Transport, passenger car, gasoline, Large SUV,...",RER,kilometer
5,"Transport, passenger car, gasoline, Medium, 20...",RER,kilometer
6,"Transport, passenger car, gasoline hybrid, Com...",RER,kilometer
7,"Transport, passenger car, gasoline, Compact, 2...",RER,kilometer
8,"Transport, passenger car, plugin gasoline hybr...",RER,kilometer
9,"Transport, passenger car, gasoline hybrid, Med...",RER,kilometer


Geography filters are often more reliable than free-text search alone. For example, you can ask for passenger-car activities in Switzerland (`CH`), Europe (`RER`), or global datasets (`GLO`).

In [71]:
car_by_location = [
    act
    for act in db
    if 'passenger car' in act['name'].lower()
    and act.get('location') in {'CH', 'RER'}
    and act["unit"] == "kilometer"
]

pd.DataFrame(
    [
        {
            'name': act['name'],
            'location': act.get('location'),
            'unit': act.get('unit'),
        }
        for act in car_by_location[:15]
    ]
)

,name,location,unit
0,"Transport, passenger car, compressed gas, Large, 2003, EURO-3",CH,kilometer
1,"Transport, passenger car, compressed gas, Medium, 2016, EURO-6ab",RER,kilometer
2,"Transport, passenger car, compressed gas, 100% biomethane, Compact, 2008, EURO-4",RER,kilometer
3,"Transport, passenger car, diesel, Compact, 2021, EURO-6d",RER,kilometer
4,"Transport, passenger car, plugin gasoline hybrid, Compact, 2016, EURO-6ab",RER,kilometer
5,"Transport, passenger car, diesel hybrid, Large SUV, 2021, EURO-6d",CH,kilometer
6,"Transport, passenger car, battery electric, NMC-622 battery, Micro, 2021",CH,kilometer
7,"Transport, passenger car, gasoline, Medium, 2008, EURO-4",CH,kilometer
8,"Transport, passenger car, plugin gasoline hybrid, label-certified electricity, Large SUV, 2016, EURO-6ab",CH,kilometer
9,"Transport, passenger car, plugin diesel hybrid, label-certified electricity, Large SUV, 2019, EURO-6d-TEMP",CH,kilometer


Let's choose one activity relating to driving a gasoline car and inspect its metadata and exchanges. Here we look at an `operation` activity, because it gives a nice mix of technosphere input(s) and biosphere emissions.

In [63]:
selected_activity = gasoline_car_candidates[0]

pprint(selected_activity.as_dict())

{'code': '386420.0',
 'comment': 'Driving cycle: WLTC. Combustion engine power: 51 [kW]. Electric '
            'motor power: 124 [kW]. Power share from combustion engine: 29 '
            '[%]. Km over lifetime: 200000 [km]. Yearly mileage: 12000 '
            '[km/year]. Autonomy on a full tank/battery: 841 [km]. '
            'Tank-to-wheel efficiency: 54 [%]. Tank-to-wheel energy '
            'consumption: 1562 [kj/km]. Battery capacity: 16 [kWh]. Fuel tank '
            'capacity: 644 [kWh]. Mass of battery: 157 [kg]. Curb mass (excl. '
            'driver and cargo): 2354 [kg]. Driving mass (incl. driver and '
            'cargo): 2494 [kg]. Electric utility factor: 47 [%]..',
 'database': 'bafu',
 'filename': 'process_3525e5f5-b77c-379f-83ce-83569e395ae6.xml',
 'id': 293107469658701824,
 'location': 'RER',
 'name': 'Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, '
         'EURO-6d',
 'reference product': 'Transport, passenger car, plugin gasoline hybrid, La

Once you have one activity, you can iterate through its exchanges. Brightway lets you inspect `production()`, `technosphere()`, and `biosphere()` separately.

In [68]:
def preview_exchanges(exchanges, n=5):
    rows = []
    for exc in list(exchanges)[:n]:
        rows.append(
            {
                'input': exc.input['name'],
                'amount': exc['amount'],
                'unit': exc.input.get('unit'),
                'location': exc.input.get('location'),
                'categories': '::'.join(exc.input.get('categories') or ()),
                'type': exc['type'],
            }
        )
    return pd.DataFrame(rows)

print('Production exchanges')
display(preview_exchanges(selected_activity.production(), n=3))

print('Technosphere exchanges')
display(preview_exchanges(selected_activity.technosphere(), n=5))

print('Biosphere exchanges')
display(preview_exchanges(selected_activity.biosphere(), n=8))

Production exchanges


,input,amount,unit,location,categories,type
0,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d",1,kilometer,RER,,production


Technosphere exchanges


,input,amount,unit,location,categories,type
0,"Brake wear emissions, passenger car",0.000011,kilogram,RER,,technosphere
1,"Carbon dioxide liquid, at plant",0.000008,kilogram,RER,,technosphere
2,"Disposal, road",0.001260,meter-year,RER,,technosphere
3,"Electricity, low voltage, production ENTSO-E, at grid",0.125100,kilowatt hour,ENTSO-E,,technosphere
4,Fuel supply for gasoline vehicles,0.049200,kilogram,RER,,technosphere


Biosphere exchanges


,input,amount,unit,location,categories,type
0,1-Pentene,1.070000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
1,1-Pentene,3.560000e-10,kilogram,None,air::non-urban air or from high stacks,biosphere
2,1-Pentene,1.230000e-09,kilogram,None,air::urban air close to ground,biosphere
3,Acetaldehyde,8.540000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
4,Acetaldehyde,2.850000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
5,Acetaldehyde,9.860000e-09,kilogram,None,air::urban air close to ground,biosphere
6,Acetone,6.410000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere
7,Acetone,2.140000e-09,kilogram,None,air::non-urban air or from high stacks,biosphere


You can use the same idea for any activity attributes (classifications, authors, year of publication), provided those are filled in. 

## Checkpoint 1

Pick one activity from the imported database, store it in `selected_activity`, and inspect a few production, technosphere, and biosphere exchanges.

Suggestion: start with a gasoline passenger car activity and refine by `location` if you get too many hits.

In [ ]:
# TODO
# Example pattern:
# hits = [
#     act for act in db
#     if 'passenger car' in act['name'].lower()
#     and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
# ]
# selected_activity = hits[0]
# preview_exchanges(selected_activity.technosphere(), n=3)
# preview_exchanges(selected_activity.biosphere(), n=3)

In [ ]:
if database_name not in bd.databases:
    print('Run the import cell first.')
else:
    db = bd.Database(database_name)
    hits = [
        act
        for act in db
        if 'passenger car' in act['name'].lower()
        and any(term in act['name'].lower() for term in ['gasoline', 'petrol'])
        and 'operation' in act['name'].lower()
    ]

    selected_activity = hits[0] if hits else db.random()

    print('Selected activity:', selected_activity['name'])
    print('Location:', selected_activity.get('location'))
    print('Unit:', selected_activity.get('unit'))

    print('First production exchanges:')
    display(preview_exchanges(selected_activity.production(), n=3))

    print('First technosphere exchanges:')
    display(preview_exchanges(selected_activity.technosphere(), n=3))

    print('First biosphere exchanges:')
    display(preview_exchanges(selected_activity.biosphere(), n=5))

## Checkpoint 2

Choose one of the LCIA methods already available in the installed project, ideally the climate-change indicator, and store it in `selected_method`.


In [ ]:
# TODO
# Example pattern:
# [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()][:5]
# selected_method = ...


In [72]:
preferred_method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

if preferred_method in bd.methods:
    selected_method = preferred_method
else:
    climate_methods = [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()]
    selected_method = climate_methods[0] if climate_methods else next(iter(bd.methods))

print('Selected method:', selected_method)
print('Method unit:', bd.Method(selected_method).metadata.get('unit'))


Selected method: ('EF v3.1', 'climate change', 'global warming potential (GWP100)')
Method unit: kg CO2-Eq


## 7) Run your first LCA and inspect the matrices

We now combine the selected activity and method into a complete LCA calculation, and then look at the matrix objects that Brightway builds behind the scenes.

In [73]:
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

# bw2calc.LCA is the LCA calculation engine of `bw2calc`
# it expect a functional unit demand, and an LCIA method (optional)
lca = bc.LCA({selected_activity: 1}, selected_method)

# here, we solve the system (As=f) and obtain the inventory
lca.lci()

# and we multiply the inventory by the characterization matrix
lca.lcia()

print('Activity:', selected_activity['name'])
print('Method:', selected_method)
print('LCA score:', lca.score)


Activity: Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d
Method: ('EF v3.1', 'climate change', 'global warming potential (GWP100)')
LCA score: 0.3349837428308963


### 7.1 From the demand dict to the demand array

The argument passed to `bc.LCA(...)` is a Python dictionary: it is easy to read, but not yet in matrix form.  
After `lca.lci()`, Brightway has translated this demand into a numerical vector in product space: `lca.demand_array`.

In [74]:
print('Demand dict:')
print(lca.demand)
print()
print('Demand array shape:', lca.demand_array.shape)
print('Number of non-zero entries in demand_array:', np.count_nonzero(lca.demand_array))

Demand dict:
{293107469658701824: 1}

Demand array shape: (11747,)
Number of non-zero entries in demand_array: 1


In [88]:
demand_id = list(lca.demand.keys())[0]
demand_id

293107469658701824

In [85]:
# where is that demand in the demand vector?
demand_index = int(np.flatnonzero(lca.demand_array)[0])
demand_index

9791

In [86]:
# we have some reversed dictionaries that can 
# help us map indices (meaning position in a vector or matrix) to IDs
lca.dicts.product.reversed

{0: 293107401207660544,
 1: 293107401211854848,
 2: 293107401211854849,
 3: 293107401216049152,
 4: 293107401216049153,
 5: 293107401220243456,
 6: 293107401220243457,
 7: 293107401220243458,
 8: 293107401249603584,
 9: 293107401253797888,
 10: 293107401253797889,
 11: 293107401257992192,
 12: 293107401257992193,
 13: 293107401262186496,
 14: 293107401278963712,
 15: 293107401278963713,
 16: 293107401283158016,
 17: 293107401283158017,
 18: 293107401283158018,
 19: 293107401287352320,
 20: 293107401287352321,
 21: 293107401291546624,
 22: 293107401291546625,
 23: 293107401291546626,
 24: 293107401295740928,
 25: 293107401295740929,
 26: 293107401299935232,
 27: 293107401312518144,
 28: 293107401312518145,
 29: 293107401316712448,
 30: 293107401316712449,
 31: 293107401316712450,
 32: 293107401320906752,
 33: 293107401320906753,
 34: 293107401320906754,
 35: 293107401325101056,
 36: 293107401325101057,
 37: 293107401325101058,
 38: 293107401329295360,
 39: 293107401329295361,
 40: 29310

In [91]:
# let's check: is the position of our demand in the demand vector 
# leads to the ID we fetched from `lca.demand`?
lca.dicts.product.reversed[demand_index] == demand_id
# Cool!

True

In [92]:
# hence, we now know who the functional unit is...
demanded_product = bd.get_activity(demand_id)
pd.Series(
    {
        'demanded product index': demand_index,
        'demanded product': demanded_product['name'],
        'location': demanded_product.get('location'),
        'amount in demand_array': lca.demand_array[demand_index],
    }
)

demanded product index                                                                          9791
demanded product          Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d
location                                                                                         RER
amount in demand_array                                                                           1.0
dtype: object

Mapping dicitonaries and their reversed forms are available for **products**, **activities** and **biosphere flows**.  
They allow us to know who is behind a specific position in a specific vector or matrix.

In [96]:
list(lca.dicts.product.reversed.items())[:5]

[(0, 293107401207660544),
 (1, 293107401211854848),
 (2, 293107401211854849),
 (3, 293107401216049152),
 (4, 293107401216049153)]

In [97]:
list(lca.dicts.activity.reversed.items())[:5]

[(0, 293107401207660544),
 (1, 293107401211854848),
 (2, 293107401211854849),
 (3, 293107401216049152),
 (4, 293107401216049153)]

In [98]:
list(lca.dicts.biosphere.reversed.items())[:5]

[(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]

### 7.2 Dictionaries that map matrix indices back to Brightway objects

The matrices only know row and column numbers. To go back from those numbers to Brightway nodes, use the dictionaries stored on the LCA object.

As seen above, those giving `id` -> `index` live in `lca.dicts.activity`, `lca.dicts.product`, and `lca.dicts.biosphere`.  
And those giving `index` -> `id` live in `lca.dicts.activity.reversed`, `lca.dicts.product.reversed`, and `lca.dicts.biosphere.reversed`

In [102]:
selected_activity_index = lca.dicts.activity[selected_activity.id]

pd.DataFrame(
    [
        {
            'dictionary': 'activity',
            'size': len(lca.dicts.activity),
            'example matrix index': selected_activity_index,
            'example node name': bd.get_node(id=lca.dicts.activity.reversed[selected_activity_index])['name'],
        },
        {
            'dictionary': 'product',
            'size': len(lca.dicts.product),
            'example matrix index': demand_index,
            'example node name': bd.get_node(id=lca.dicts.product.reversed[demand_index])['name'],
        },
        {
            'dictionary': 'biosphere',
            'size': len(lca.dicts.biosphere),
            'example matrix index': 0,
            'example node name': bd.get_node(id=lca.dicts.biosphere.reversed[0])['name'],
        },
    ]
)

,dictionary,size,example matrix index,example node name
0,activity,11747,9791,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d"
1,product,11747,9791,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d"
2,biosphere,1807,0,"1,4-Butanediol"


### 7.3 The technosphere matrix and the biosphere matrix

After `lca.lci()`, Brightway has built:

- `lca.technosphere_matrix`: the technosphere matrix $A$
- `lca.biosphere_matrix`: the biosphere matrix $B$

For a demanded product, a column of the technosphere matrix shows the products consumed and produced by that unit process. A column of the biosphere matrix shows the direct elementary flows of the corresponding activity.

In [104]:
pd.DataFrame(
    [
        {'object': 'demand_array', 'shape': lca.demand_array.shape},
        {'object': 'technosphere_matrix', 'shape': lca.technosphere_matrix.shape},
        {'object': 'biosphere_matrix', 'shape': lca.biosphere_matrix.shape},
        {'object': 'supply_array', 'shape': lca.supply_array.shape},
        {'object': 'inventory', 'shape': lca.inventory.shape},
        {'object': 'characterization_matrix', 'shape': lca.characterization_matrix.shape},
        {'object': 'characterized_inventory', 'shape': lca.characterized_inventory.shape},
    ]
)

,object,shape
0,demand_array,"(11747,)"
1,technosphere_matrix,"(11747, 11747)"
2,biosphere_matrix,"(1807, 11747)"
3,supply_array,"(11747,)"
4,inventory,"(1807, 11747)"
5,characterization_matrix,"(1807, 1807)"
6,characterized_inventory,"(1807, 11747)"


In [107]:
# Let's slice the technosphere matrix to fetch all the rows (products) 
# at the column corresponding to our gasoline car transport activity.
technosphere_column = lca.technosphere_matrix[:, demand_index].tocoo()
technosphere_column

<COOrdinate sparse matrix of dtype 'float64'
	with 14 stored elements and shape (11747, 1)>

In [109]:
# let's map those indices to product names and locations
technosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'product': bd.get_node(id=rev_prod[row_idx])['name'],
            'location': bd.get_node(id=rev_prod[row_idx]).get('location'),
            'amount in A': amount,
        }
        for row_idx, amount in zip(technosphere_column.row, technosphere_column.data)
    ]
)

technosphere_view.sort_values('amount in A', ascending=False)

# notice how inputs are stored as negative entries in the technosphere/A matrix
# except the outgoing/production exchange, which is positive and on the diagonal (where row index == col index)

,row index,col index,product,location,amount in A
12,9791,9791,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d",RER,1.000000e+00
10,8741,9791,"Transport, freight, lorry, 16t-32t gross weight, fleet average",RER,-8.020000e-07
11,9113,9791,"Transport, freight, rail",RER,-4.810000e-06
7,7239,9791,"Passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d",RER,-5.000000e-06
1,532,9791,"Carbon dioxide liquid, at plant",RER,-8.020000e-06
0,441,9791,"Brake wear emissions, passenger car",RER,-1.110000e-05
5,5087,9791,"Maintenance, passenger car",RER,-1.270000e-05
9,7818,9791,"Road wear emissions, passenger car",RER,-2.230000e-05
13,9955,9791,"Tyre wear emissions, passenger car",RER,-2.450000e-05
2,1818,9791,"Disposal, road",RER,-1.260000e-03


In [110]:
# here too, let's slice the biosphere matrix to fetch all the rows (elementary flows) 
# at the column corresponding to our gasoline car transport activity.
biosphere_column = lca.biosphere_matrix[:, selected_activity_index].tocoo()

biosphere_view = pd.DataFrame(
    [
        {
            'row index': row_idx,
            'col index': demand_index,
            'biosphere flow': bd.get_node(id=rev_bio[row_idx])['name'],
            'categories': '::'.join(bd.get_node(id=rev_bio[row_idx]).get('categories') or ()),
            'amount in B': amount,
        }
        for row_idx, amount in zip(biosphere_column.row, biosphere_column.data)
    ]
)

biosphere_view.sort_values('amount in B', ascending=False).head(12)

,row index,col index,biosphere flow,categories,amount in B
44,637,9791,"Carbon dioxide, fossil",air,1.483131e-01
43,634,9791,"Carbon dioxide, non-fossil",air,3.857280e-03
45,642,9791,"Carbon monoxide, fossil",air::non-urban air or from high stacks,1.375000e-04
9,83,9791,"Carbon monoxide, fossil",air::urban air close to ground,2.670000e-05
76,1765,9791,Nitrogen oxides,air::non-urban air or from high stacks,8.110000e-06
38,584,9791,Ammonia,air::non-urban air or from high stacks,6.300000e-06
53,729,9791,"Hydrocarbons, chlorinated",air::non-urban air or from high stacks,2.409000e-06
17,223,9791,"Hydrocarbons, chlorinated",air::urban air close to ground,1.430000e-06
75,1764,9791,Nitrogen oxides,air::urban air close to ground,1.420000e-06
4,28,9791,Ammonia,air::urban air close to ground,9.370000e-07


For this notebook, two practical readings are enough:

- in the technosphere column, the positive diagonal entry is the reference product produced by the activity
- the other entries are the technosphere inputs used by that activity
- in the biosphere column, each row is a direct emission or resource use of that activity before upstream scaling

### 7.4 Supply array and inventory

`lca.supply_array` indicates how much each activity must run to meet the final demand $$\mathbf{A}\mathbf{s} = \mathbf{f}$$  

In [112]:
# let's print the activities and the amount they must supply to satisfy the functional unit.
nonzero_supply = np.flatnonzero(lca.supply_array)

print(f"Number of supplying activities throughout the system: {nonzero_supply.shape}")

supply_view = pd.DataFrame(
    [
        {
            'activity': bd.get_node(id=lca.dicts.activity.reversed[idx])['name'],
            'location': bd.get_node(id=lca.dicts.activity.reversed[idx]).get('location'),
            'amount in supply_array': lca.supply_array[idx],
        }
        for idx in nonzero_supply
    ]
)

# let's print only the first 12 ones
supply_view.sort_values('amount in supply_array', ascending=False).head(12)

# These are 12 out of many activities along the supply chain that must provide an output to satisfy the functional unit

Number of supplying activities throughout the system: (4215,)


,activity,location,amount in supply_array
3924,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d",RER,1.000000
4091,"Water, decarbonised, at plant",RER,0.450505
3941,"Transport, transoceanic tanker",OCE,0.363894
721,"Disposal, spoil from lignite mining, in surface landfill",GLO,0.198247
1300,"Electricity, production mix",ENTSO-E,0.189245
985,"Electricity, high voltage, production ENTSO-E, at grid",ENTSO-E,0.184057
1128,"Electricity, medium voltage, production ENTSO-E, at grid",ENTSO-E,0.179731
1501,"Gravel, crushed, at mine",CH,0.154211
2191,"Natural gas, high pressure, at consumer",FR,0.151485
2166,"Natural gas, high pressure, at consumer",CH,0.138851


The inventory matrix then scales `lca.biosphere_matrix` by `lca.supply_array` $$\mathbf{G} = \mathbf{B}\,\mathrm{diag}(\mathbf{s})$$  
So summing a row in `lca.biosphere_matrix` gives a cradle-to-gate total for one pollutant.

In [116]:
noxs = [
    flow
    for flow in bd.Database('ecoinvent-3.10-biosphere')
    if flow['name'] == 'Nitrogen oxides'
]
noxs

['Nitrogen oxides' (kilogram, None, ('air', 'low population density, long-term')),
 'Nitrogen oxides' (kilogram, None, ('air', 'urban air close to ground')),
 'Nitrogen oxides' (kilogram, None, ('air',)),
 'Nitrogen oxides' (kilogram, None, ('air', 'non-urban air or from high stacks')),
 'Nitrogen oxides' (kilogram, None, ('air', 'lower stratosphere + upper troposphere'))]

In [122]:
[nox.id for nox in noxs]

[4199, 4197, 4201, 4198, 4200]

But, let' be careful here: `lca.biosphere_matrix` isn't as large as the `biosphere` database, because `matrix_utils` (a sub-library) only fill in `lca.biosphere_matrix` with flows that are actuall used.

In [124]:
len(bd.Database('ecoinvent-3.10-biosphere'))

4362


In [125]:
lca.biosphere_matrix.shape

(1807, 11747)

In [127]:
nox_rows = [lca.dicts.biosphere[nox.id] for nox in noxs if nox.id in lca.dicts.biosphere]
nox_rows

[1764, 1767, 1765, 1766]

In [132]:
print("NOx flow id", " | ", "NOx index in lca.biosphere_matrix")
for flow, flow_idx in zip(noxs, nox_rows):
    print(flow.id, " | ", flow_idx)

NOx flow id  |  NOx index in lca.biosphere_matrix
4199  |  1764
4197  |  1767
4201  |  1765
4198  |  1766


In [143]:
emissions = []
for row in nox_rows:
    flow = bd.get_activity(lca.dicts.biosphere.reversed[row])
    emissions.append(
        [
            flow["name"],
            flow["categories"],
            flow["unit"],
            lca.inventory[row, selected_activity_index]
        ]
    )

print(f"Cradle-to-gate NOx emissions for activity with index {selected_activity_index}.")
pd.DataFrame(emissions, columns=["name", "categories", "unit", "amount"])

Cradle-to-gate NOx emissions for activity with index 9791.


,name,categories,unit,amount
0,Nitrogen oxides,"(air, urban air close to ground)",kilogram,0.000001
1,Nitrogen oxides,"(air,)",kilogram,0.000000
2,Nitrogen oxides,"(air, non-urban air or from high stacks)",kilogram,0.000008
3,Nitrogen oxides,"(air, lower stratosphere + upper troposphere)",kilogram,0.000000


### 7.5 Characterized inventory

After `lca.lcia()`, Brightway applies the chosen characterization factors. The matrix `lca.characterized_inventory` has the same shape as `lca.inventory`, but its values are now in impact-score units rather than physical flow units. Summing all its cells gives the LCIA score.

In [145]:
print('LCA score:', lca.score)

LCA score: 0.3349837428308963


In [149]:
print('LCA score:', np.sum(lca.characterization_matrix @ lca.inventory))

LCA score: 0.3349837428308963


Let's sum `lca.characterized_inventory` column-wise (we collapsed activities)

In [ ]:
characterized_by_flow = np.asarray(lca.characterized_inventory.sum(axis=1)).ravel()

In [158]:
# let's sort them by individual score and remove those that are zero and keep teh ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_flow) 
    if characterized_by_flow[idx] != 0
]

# let's revert it, because np.argsort sorts ascendingly
top_flow_indices = top_flow_indices[::-1]
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.biosphere.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.biosphere.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_flow[idx],
        }
        for idx in top_flow_indices
    ]
)

,biosphere flow,categories,characterized contribution
0,"Carbon dioxide, fossil",air,0.181037
1,"Carbon dioxide, fossil",air::urban air close to ground,0.085762
2,"Carbon dioxide, fossil",air::non-urban air or from high stacks,0.028348
3,"Methane, fossil",air,0.027267
4,"Methane, fossil",air::non-urban air or from high stacks,0.004413
5,Sulfur hexafluoride,air::non-urban air or from high stacks,0.002585
6,"Methane, fossil",air::urban air close to ground,0.001116
7,Dinitrogen monoxide,air::urban air close to ground,0.000831
8,Sulfur hexafluoride,air,0.000734
9,"Methane, non-fossil",air::non-urban air or from high stacks,0.000657


We can also sum row-wise to show the contribution of activities.

In [157]:
characterized_by_activity = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()

In [161]:
# let's sort them by individual score and remove those that are zero and keep teh ten largest contributors
top_flow_indices = [
    idx for idx 
    in np.argsort(characterized_by_activity) 
    if characterized_by_activity[idx] != 0
]

# let's revert it, because np.argsort sorts ascendingly
top_flow_indices = top_flow_indices[::-1]
# and let's pick the ten largest
top_flow_indices = top_flow_indices[:10]

pd.DataFrame(
    [
        {
            'biosphere flow': bd.get_activity(id=lca.dicts.activity.reversed[idx])['name'],
            'categories': '::'.join(bd.get_activity(id=lca.dicts.activity.reversed[idx]).get('categories') or ()),
            'characterized contribution': characterized_by_activity[idx],
        }
        for idx in top_flow_indices
    ]
)

,biosphere flow,categories,characterized contribution
0,"Transport, passenger car, plugin gasoline hybrid, Large SUV, 2021, EURO-6d",,0.148372
1,"Natural gas, vented",,0.025795
2,"Disposal, residues, shredder fraction from manual dismantling, in MSWI",,0.009364
3,"Disposal, plastics, mixture, 15.3% water, to municipal incineration",,0.008196
4,"Lignite, burned in power plant",,0.006571
5,"Sweet gas, burned in gas turbine, production",,0.005911
6,"Diesel, burned in building machine, average",,0.005277
7,"Natural gas, burned in industrial furnace 1MWth",,0.005135
8,"Hard coal, burned in power plant",,0.004896
9,"Pig iron, at plant",,0.004566


## 8) Dedicated `ecoinvent` import overview

In this notebook, we only use the prepared `ecoinvent-3.10-biosphere` project archive as a shortcut for `biosphere3` and the LCIA methods.  
We do not import the `ecoinvent` technosphere database itself.

Key points:
- `ecoinvent` is not distributed in this repository.
- The full `ecoinvent` technosphere import is not run during this course.
- The overall logic is the same: create an importer, apply strategies, inspect statistics, then write the database.
- In practice, `ecoinvent` uses dedicated `bw2io` importers.


In [ ]:
# Example only. Do not run in this course.

# bi.import_ecoinvent_release(
#     version="3.10", 
#     system_model="cutoff", # other options are "consequential", "apos" and "EN15804"
#     username="xxxx",
#     password="xxxx",
#     biosphere_name="biosphere" # optional, otherwise a name is chosen for you
# )

## 9) Brief troubleshooting notes

- Project install fails: the first run of `bi.remote.install_project(...)` needs an internet connection to download the archive from `https://files.brightway.dev/`.
- Project already exists: the setup cell switches to the existing local project instead of downloading it again.
- Incomplete project contents: if the project exists but is missing `biosphere3` or the LCIA methods, delete that project or choose a fresh project name and rerun section 1.
- Workbook not found: confirm that `../../data/lci-bafu.xlsx` exists.
- `bafu-2025` missing: rerun the BAFU import cells so that the workbook is parsed and written to the project.


## Recap

After this notebook, you should now know how to:

- bootstrap a `brightway` project from a prepared archive
- import a database with `ExcelImporter`
- inspect the LCIA methods already available in a project
- import your own LCIA method
- interpret the role of strategies, matching, and statistics
- search a database for activities and inspect exchanges
- run a first LCA from demand to score
- read `demand_array`, `technosphere_matrix`, `biosphere_matrix`, `supply_array`, and `inventory`
- use the LCA dictionaries to move between matrix indices and Brightway objects
- extract cradle-to-gate pollutant totals and interpret `characterized_inventory`
- recognize how the same import logic would extend to `ecoinvent`